# 02 - Preprocessing e raggruppamento ICD-9

## Obiettivi didattici

1. Implementare il mapping **ICD-9 -> 9 macro-categorie** secondo Strack et al. (2014, Tabella 2).
2. Costruire le **feature derivate cliniche**: complessita' farmacologica, intensita' di utilizzo sanitario pregresso, comorbidita'.
3. Configurare il `ColumnTransformer` (imputer mediana per numeriche, OneHotEncoder con `min_frequency` per categoriche).
4. Verificare la struttura della pipeline serializzabile.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

from readmit_pipeline.data import load_raw, basic_clean
from readmit_pipeline.icd9 import map_icd9_to_category, ALL_CATEGORIES
from readmit_pipeline.features import ReadmissionFeatureEngineer
from readmit_pipeline.preprocessing import build_preprocessor, infer_column_groups


## Mapping ICD-9 -> macro-categorie (Strack 2014)


In [ ]:
examples = ['250.83', '428.0', '486', '530', 'V58.67', 'E885.9', '999', '710']
for c in examples:
    print(f'  {c:>8s} -> {map_icd9_to_category(c)}')
print()
print('Categorie disponibili:', ALL_CATEGORIES)


## Feature engineering


In [ ]:
df = basic_clean(load_raw())
fe = ReadmissionFeatureEngineer()
df_fe = fe.fit_transform(df.drop(columns=['readmitted', 'readmitted_30d']))
added = [c for c in df_fe.columns if c not in df.columns]
print('Feature derivate aggiunte:')
for c in added:
    print(f'  - {c}')


## Distribuzione delle macro-categorie diagnostiche


In [ ]:
if 'diag_1_cat' in df_fe.columns:
    df_fe['diag_1_cat'].value_counts().plot(kind='bar')
    plt.title('diag_1: macro-categoria principale')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout(); plt.show()


## ColumnTransformer

Combina:
- imputer mediana + StandardScaler sulle feature numeriche.
- imputer 'Unknown' + OneHotEncoder (`min_frequency=10`) sulle categoriche.


In [ ]:
groups = infer_column_groups(df_fe)
print({k: len(v) for k, v in groups.items()})
preprocessor = build_preprocessor(groups['numeric'], groups['categorical'])
preprocessor
